# Single-profile trajectory VAE
Adapted from **BO_test V2.ipynb**. One input trajectory and one reconstructed trajectory: `[t, Profile]`, each with `SEQ_LEN` control points. Model tensors have shape `(batch, 2, SEQ_LEN)`; the two rows are coordinates of one profile, not two independent profiles.

Run cells in order. Set `PROFILE_KIND` to `"T"` or `"AS"` to select the synthetic training family. The encoder, decoder, loss and BO helpers all use the same single-profile interface. This standalone notebook retains the original VAE architecture and training approach; unrelated CSV evaluation and one-dimensional BO demonstrations are omitted.

Time must increase strictly and be normalized from 0 to 1. Profile values must be scaled to [0, 1] using consistent physical bounds. Decoding produces normalized coordinates; convert back to physical units before use. Existing four-row model checkpoints are incompatible and the model must be retrained.

Dependencies: `numpy`, `matplotlib`, `torch`. If needed, run `%pip install numpy matplotlib torch` in a separate cell and restart the kernel.

In [ ]:
import math
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROFILE_KIND = "T"  # "T": cooling ramps/holds; "AS": addition steps/holds
SEQ_LEN = 8
LATENT_DIM = 7
BATCH_SIZE = 64
N_sampl = 20000
N_sampl_VAL = 2000
EPOCHS = 300
BETA_MAX = 1e-3
KL_WARMUP = 30
assert PROFILE_KIND in ("T", "AS")
print("Device:", DEVICE)

## Encoder and single-profile decoder
Fourier features, residual blocks and latent sampling are retained. The decoder has one value head and one time-interval head. Positive intervals give ordered time coordinates from 0 to 1; values use a sigmoid.

In [ ]:
# ================================================================
# 1. Fourier Feature Encoding
# ================================================================
class PositionalEncoding1D(nn.Module):
    def __init__(self, num_bands=2):
        super().__init__()
        self.num_bands = num_bands

    def forward(self, x):
        # x: (batch, seq_len)
        out = [x]
        for i in range(self.num_bands):
            freq = (2.0 ** i) * math.pi
            out.append(torch.sin(freq * x))
            out.append(torch.cos(freq * x))
        return torch.cat(out, dim=-1)


# ================================================================
# 2. Residual Block
# ================================================================
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return x + 0.5 * self.net(x)


# ================================================================
# 3. Single-profile Trajectory VAE
# ================================================================
class ImprovedTrajectoryVAE(nn.Module):
    def __init__(
        self,
        seq_len=8,
        latent_dim=4,
        hidden_dims=(64, 32),
        num_bands=2
    ):
        super().__init__()

        self.seq_len = seq_len
        self.latent_dim = latent_dim

        h1, h2 = hidden_dims

        self.pos_enc = PositionalEncoding1D(num_bands)
        feat_dim = 1 + 2 * num_bands
        input_dim = 2 * seq_len * feat_dim

        # ---------------- ENCODER ----------------
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.LayerNorm(h1),
            nn.SiLU(),
            ResidualBlock(h1),
            nn.Linear(h1, h2),
            nn.LayerNorm(h2),
            nn.SiLU()
        )

        self.fc_mu = nn.Linear(h2, latent_dim)
        self.fc_logvar = nn.Linear(h2, latent_dim)

        # ---------------- DECODER ----------------
        # ========================================================
        # Shared decoder backbone
        # ========================================================
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, h2),
            nn.SiLU(),
            nn.Linear(h2, h1),
            nn.SiLU(),
            ResidualBlock(h1)
        )        
        # Profile profile
        self.decoder_profile = nn.Sequential(
            nn.Linear(h1, h2),
            nn.SiLU(),
            nn.Linear(h2, seq_len),
            nn.Sigmoid()
        )
        
        # Profile time coordinates
        self.decoder_t_profile = nn.Sequential(
            nn.Linear(h1, h2),
            nn.SiLU(),
            nn.Linear(h2, seq_len - 1)
        )
        


    def encode(self, x):
        # x: (batch, 2, seq_len)
        b, c, s = x.shape

        if c != 2 or s != self.seq_len:
            raise ValueError(
                f"Expected (batch, 2, {self.seq_len}), "
                f"received {tuple(x.shape)}"
            )

        x_flat = x.reshape(b * c, s)
        x_encoded = self.pos_enc(x_flat)
        h = x_encoded.reshape(b, -1)

        h = self.encoder(h)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        if not self.training:
            return mu

        logvar = torch.clamp(logvar, -12, 8)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

        
    
    def decode(self, z):
    
        h = self.decoder(z)
    
        # ========================================================
        # Profile profile
        # ========================================================
        recon_profile = self.decoder_profile(h)
    
        # Positive time intervals for profile
        raw_dt = self.decoder_t_profile(h)
        dt = F.softplus(raw_dt) + 1e-5
    
        # Normalize total duration to 1
        dt = dt / dt.sum(dim=-1, keepdim=True)
    
        zeros = torch.zeros_like(dt[:, :1])
    
        recon_t = torch.cat(
            [zeros, torch.cumsum(dt, dim=-1)],
            dim=-1
        )
    
        return torch.stack([recon_t, recon_profile], dim=1)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

## Single-profile loss
Retains reconstruction, interval, transition, gradient, endpoint and free-bits KL penalties. Curvature is optional. Only the selected profile contributes to the loss.

In [ ]:
def free_bits_kl_loss(mu, logvar, threshold=0.05):
    logvar = logvar.clamp(-12, 8)
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).mean(dim=0)
    return kl_per_dim.clamp_min(threshold).sum()


def vae_loss_function(recon_x, target, mu, logvar, beta=1e-3,
                      gamma_profile=0.1, delta_profile=0.0, alpha=2.0,
                      w_time=1.0, w_profile=1.5, w_dt=0.01,
                      switch_weight=3.0, w_switch=1.0,
                      return_components=False):
    if recon_x.shape != target.shape or target.ndim != 3 or target.shape[1] != 2:
        raise ValueError("Reconstruction and target must both have shape (batch, 2, seq_len)")
    t_true, p_true = target[:, 0], target[:, 1]
    t_pred, p_pred = recon_x[:, 0], recon_x[:, 1]
    dp_true, dp_pred = torch.diff(p_true), torch.diff(p_pred)
    dt_true, dt_pred = torch.diff(t_true), torch.diff(t_pred)
    weights = (1.0 + switch_weight * dp_true.abs()).detach()

    def weighted_mean(error, weight):
        return ((error * weight).sum(-1) / weight.sum(-1).clamp_min(1e-8)).mean()

    interval_loss = weighted_mean((dt_pred - dt_true).abs(), weights)
    interval_loss += 0.1 * weighted_mean(F.smooth_l1_loss(
        dt_pred.clamp_min(1e-6).log(), dt_true.clamp_min(1e-6).log(),
        beta=0.5, reduction="none"), weights)
    midpoint_true = 0.5 * (t_true[:, :-1] + t_true[:, 1:])
    midpoint_pred = 0.5 * (t_pred[:, :-1] + t_pred[:, 1:])
    curvature = (F.l1_loss(torch.diff(p_pred, n=2), torch.diff(p_true, n=2))
                 if delta_profile > 0 and p_true.shape[-1] >= 3 else recon_x.new_zeros(()))
    components = {
        "reconstruction": w_time * F.smooth_l1_loss(t_pred, t_true, beta=0.05)
                          + w_profile * F.smooth_l1_loss(p_pred, p_true, beta=0.05),
        "intervals": w_dt * interval_loss,
        "gradient": gamma_profile * weighted_mean((dp_pred - dp_true).abs(), weights),
        "switch": w_switch * weighted_mean((midpoint_pred - midpoint_true).abs(), dp_true.abs().detach()),
        "curvature": delta_profile * curvature,
        "endpoints": alpha * w_profile * (F.mse_loss(p_pred[:, 0], p_true[:, 0])
                                         + F.mse_loss(p_pred[:, -1], p_true[:, -1])),
        "KL": beta * free_bits_kl_loss(mu, logvar),
    }
    loss = sum(components.values())
    if return_components:
        return loss, {key: value.detach().item() for key, value in components.items()}
    return loss

## Synthetic training data
The original temperature and antisolvent generators are retained. Only the chosen generator is used, producing `(N, 2, SEQ_LEN)` data. For measured data, replace the two dataset tensors with training and validation sets normalized using the same fixed bounds.

In [ ]:
# Generate V- and AS-profiles for VAE fitting

class ASProfDef:
    def __init__(self, t_min=0, t_max=0.2, V_min=0, V_max=1,choices=['add','add','hold','hold','add','hold'],typpe=None):
        self.V_min = V_min
        self.V_max = V_max
        self.range_of_t = np.linspace(t_min, t_max, 10)
        self.range_of_V = np.linspace(V_min, V_max, 10)
        self.choices = choices
        self.seq_len = 8
        
        self.typpe = typpe
        self.order = 'hold'
        # removed: self.ramps / self.V_0 no longer fixed here

    def reset(self):
        # NEW: draw a fresh topology and start point every reset
        if self.typpe:
            self.ramps = ['hold', 'add','hold','add','hold'] +['hold']*(self.seq_len-6)
            self.V_profile = [0]
            
        else:
            self.ramps = np.random.choice( self.choices, size=self.seq_len - 1)    
            self.V_0 = float(np.random.choice(self.range_of_V, size=1)[0])
            self.V_profile = [self.V_0]
        self.t_profile = [0]
        
    def list_conv(self,t):
        if np.shape(t):
            t = t[0]
        return t
        
    def holding(self,t_h):
        t_h = self.list_conv(t_h)
        self.t_profile.append(float(t_h))
        self.V_profile.append(float(self.V_profile[-1]))

    def resample_profile(self, t_max=17.0):
        #t_uniform = np.linspace(0, max(self.t_profile), self.seq_len)
        #V_uniform = np.interp(t_uniform, self.t_profile, self.V_profile)
        self.t_profile = self.t_profile / max(self.t_profile)
        return self.t_profile * 2 -1 , self.V_profile

    def ramp(self,t_r,V_r):
        t_r = self.list_conv(t_r)
        V_r = self.list_conv(V_r)

        self.t_profile.append(float(t_r))
        self.V_profile.append(float(V_r))
        pass

    def generate(self):
        self.reset()

        for i in self.ramps:
            # Force step duration to be at least 0.05 to avoid zero-length segments
            random_duration = np.random.uniform(0.00001, 0.5)
            
            if i == 'hold':
                self.holding(random_duration)
            elif i == 'add':
                random_v = np.random.uniform(self.V_min, self.V_max)
                self.ramp(0.01, random_v)
    
        self.t_profile = np.cumsum(self.t_profile)
        # Ensure t strictly scales between [-1, 1]
        t_max = self.t_profile[-1]
        if t_max > 0:
            self.t_profile = (self.t_profile / t_max) 
        
        return self.t_profile, self.V_profile   

    def pad_profile(self, max_len=5):
        curr_len = max(np.cumsum(self.t_profile))
        if max_len > curr_len:
            self.t_profile = np.append(self.t_profile, max_len - curr_len)
            self.V_profile = np.append(self.V_profile, self.V_profile[-1])
            return self.t_profile, self.V_profile
        else:
            print('ERROR')
            pass

# Generate V- and AS-profiles for VAE fitting

class ProfDef:
    def __init__(self, t_min=0, t_max=0.2, V_min=-1, V_max=1,choices=['ramp','hold','hold','hold','ramp','hold'],typpe=None):
        self.V_min = V_min
        self.V_max = V_max
        self.range_of_t = np.linspace(t_min, t_max, 10)
        self.range_of_V = np.linspace(V_min, V_max, 10)
        self.choices = choices
        self.seq_len = 8
        self.typpe = typpe
        # removed: self.ramps / self.V_0 no longer fixed here

    def reset(self):
        
        if self.typpe:
            self.V_profile = [1.0]
            
        else:
            self.V_0 = float(np.random.choice(self.range_of_V, size=1)[0])
            self.V_profile = [self.V_0]
            
        self.ramps = np.random.choice( self.choices, size=self.seq_len - 1)    
             
        self.t_profile = [0]
        
    def list_conv(self,t):
        if np.shape(t):
            t = t[0]
        return t
        
    def holding(self,t_h):
        t_h = self.list_conv(t_h)
        self.t_profile.append(float(t_h))
        self.V_profile.append(float(self.V_profile[-1]))

    def resample_profile(self, t_max=17.0):
        #t_uniform = np.linspace(0, max(self.t_profile), self.seq_len)
        #V_uniform = np.interp(t_uniform, self.t_profile, self.V_profile)
        self.t_profile = self.t_profile / max(self.t_profile)
        return self.t_profile * 2 -1 , self.V_profile

    def ramp(self,t_r,V_r):
        t_r = self.list_conv(t_r)
        V_r = self.list_conv(V_r)
        if self.typpe:
            if t_r > 0.1:
                t_r = 0.1
        self.t_profile.append(float(t_r))
        self.V_profile.append(float(V_r))
        pass

    def generate(self):
        self.reset()

        for i in self.ramps:
            # Force step duration to be at least 0.05 to avoid zero-length segments
            random_duration = np.random.uniform(0.05, 1.0)
            
            if i == 'hold':
                self.holding(random_duration)
            elif i == 'ramp':
                random_v = np.random.uniform(self.V_min, min(self.V_profile))
                self.ramp(random_duration, random_v)
    
        self.t_profile = np.cumsum(self.t_profile)
        # Ensure t strictly scales between [-1, 1]
        t_max = self.t_profile[-1]
        if t_max > 0:
            self.t_profile = (self.t_profile / t_max) 
        
        return self.t_profile, self.V_profile   

    def pad_profile(self, max_len=5):
        curr_len = max(np.cumsum(self.t_profile))
        if max_len > curr_len:
            self.t_profile = np.append(self.t_profile, max_len - curr_len)
            self.V_profile = np.append(self.V_profile, self.V_profile[-1])
            return self.t_profile, self.V_profile
        else:
            print('ERROR')
            pass

In [ ]:
if PROFILE_KIND == "T":
    generator = ProfDef(V_min=0.0, V_max=1.0, t_min=0.2, t_max=1.0,
                        choices=['ramp', 'hold', 'hold', 'hold', 'hold', 'hold'], typpe='T')
else:
    generator = ASProfDef(V_min=0.0, V_max=1.0, t_min=0, t_max=1.0,
                          choices=['add', 'add', 'hold', 'hold', 'hold'], typpe='AS')
generator.seq_len = SEQ_LEN
if PROFILE_KIND == "AS" and SEQ_LEN < 6:
    raise ValueError("The original AS generator requires SEQ_LEN >= 6")

def make_dataset(n_samples):
    profiles = []
    for _ in range(n_samples * 100):
        t, values = generator.generate()
        if len(values) == SEQ_LEN and np.ptp(values) >= 1e-6:
            profiles.append([t, values])
        if len(profiles) == n_samples:
            return torch.tensor(np.asarray(profiles), dtype=torch.float32)
    raise RuntimeError("Could not generate enough nonconstant profiles")

synthetic_dataset = make_dataset(N_sampl)
synthetic_dataset_VAL = make_dataset(N_sampl_VAL)
print("Train:", synthetic_dataset.shape, "Validation:", synthetic_dataset_VAL.shape)

## Train and restore the best validation checkpoint
Default training size and epoch count match the original. Reduce these settings for a quick trial. Validation uses the latent mean. Checkpoint comparison begins after KL warmup, or at the last epoch for shorter trials.

In [ ]:
model = ImprovedTrajectoryVAE(seq_len=SEQ_LEN, latent_dim=LATENT_DIM,
                              hidden_dims=(64, 32), num_bands=2).to(DEVICE)

In [ ]:
# ============================================================
# Initialization
# ============================================================
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=15
)



best_val_loss = float('inf')
best_model_state = None
best_epoch = None

train_losses = []
val_losses = []

# Shared loss configuration
loss_params = dict(
    gamma_profile=0.1 if PROFILE_KIND == "T" else 0.5,
    delta_profile=0.1 if PROFILE_KIND == "T" else 0.0,
    alpha=2.0, w_time=1.0,
    w_profile=1.5 if PROFILE_KIND == "T" else 1.0,
    w_dt=0.01 if PROFILE_KIND == "T" else 0.02,
    switch_weight=3.0, w_switch=1.0 if PROFILE_KIND == "T" else 2.0,
)

# Use the same device as the model
device = next(model.parameters()).device

# Move datasets to the model's device
synthetic_dataset = synthetic_dataset.to(device)
synthetic_dataset_VAL = synthetic_dataset_VAL.to(device)

# ============================================================
# Training
# ============================================================
for epoch in range(EPOCHS):

    # KL warmup
    beta = BETA_MAX * min(
        1.0, (epoch + 1) / KL_WARMUP
    )

    model.train()

    permutation = torch.randperm(
        synthetic_dataset.size(0),
        device=device
    )

    epoch_loss = 0.0

    for i in range(
        0, synthetic_dataset.size(0), BATCH_SIZE
    ):

        indices = permutation[i:i + BATCH_SIZE]
        batch_x = synthetic_dataset[indices]

        optimizer.zero_grad()

        recon_x, mu, logvar = model(batch_x)

        # Correct training target and KL weight
        loss = vae_loss_function(
            recon_x,
            batch_x,
            mu,
            logvar,
            beta=beta,
            **loss_params
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        epoch_loss += loss.item() * batch_x.size(0)

    epoch_loss /= synthetic_dataset.size(0)

    # ========================================================
    # Validation
    # ========================================================
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for i in range(
            0, synthetic_dataset_VAL.size(0), BATCH_SIZE
        ):

            batch_val = synthetic_dataset_VAL[
                i:i + BATCH_SIZE
            ]

            recon_val, mu_val, logvar_val = model(
                batch_val
            )

            # Correct validation reconstruction and target
            v_loss = vae_loss_function(
                recon_val,
                batch_val,
                mu_val,
                logvar_val,
                beta=beta,
                **loss_params
            )

            val_loss += v_loss.item() * batch_val.size(0)

    val_loss /= synthetic_dataset_VAL.size(0)

    # ========================================================
    # Store loss history
    # ========================================================
    train_losses.append(epoch_loss)
    val_losses.append(val_loss)

    # ========================================================
    # Scheduler and best checkpoint
    # ========================================================
    # Compare models after KL warmup, when beta is fixed
    if epoch + 1 >= min(KL_WARMUP, EPOCHS):

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(
                model.state_dict()
            )

    # ========================================================
    # Logging
    # ========================================================
    if (epoch + 1) % 10 == 0 or epoch == 0:

        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"- Train Loss: {epoch_loss:.8f} "
            f"- Val Loss: {val_loss:.8f} "
            f"- Beta: {beta:.6f} "
            f"- LR: {optimizer.param_groups[0]['lr']:.6e}"
        )

# ============================================================
# Restore best model
# ============================================================
if best_model_state is not None:
    model.load_state_dict(best_model_state)

model.eval()

print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.8f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))

plt.plot(train_losses, label='Training loss')
plt.plot(val_losses, label='Validation loss')

plt.axvline(
    KL_WARMUP - 1,
    color='gray',
    linestyle='--',
    label='End of KL warmup'
)

plt.xlabel('Epoch')
plt.ylabel('VAE loss')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Latent-space helpers
BO encodes one profile to a latent vector and decodes it back to one profile. `seed_amount` remains external metadata, as in the original. These helpers return normalized coordinates, not physical operating instructions.

In [ ]:
def decode_bo_suggestion(seed_amount: float, z_latent: np.ndarray) -> dict:
    model.eval()
    device = next(model.parameters()).device

    z_latent = np.asarray(z_latent, dtype=np.float32).reshape(-1)

    if z_latent.shape != (model.latent_dim,):
        raise ValueError(
            f"Expected latent shape ({model.latent_dim},), "
            f"received {z_latent.shape}"
        )

    with torch.no_grad():
        z_tensor = torch.as_tensor(
            z_latent, dtype=torch.float32, device=device
        ).unsqueeze(0)

        decoded_profile = model.decode(z_tensor)
        profile_array = decoded_profile.squeeze(0).cpu().numpy()

    return {
        "seed_amount": seed_amount,
        "executable_profile_trajectory": profile_array,
        "t": profile_array[0],
        "Profile": profile_array[1],
    }


def encode_profile_for_bo(
    seed_amount: float,
    profile_trajectory: np.ndarray,
    use_mean_only: bool = True
) -> dict:
    model.eval()
    device = next(model.parameters()).device

    profile_array = np.asarray(
        profile_trajectory, dtype=np.float32
    )

    # Accept both unbatched and single-batch inputs
    if profile_array.shape == (2, model.seq_len):
        profile_array = profile_array[np.newaxis, :, :]

    elif profile_array.shape != (1, 2, model.seq_len):
        raise ValueError(
            f"Expected (2, {model.seq_len}) or "
            f"(1, 2, {model.seq_len}), "
            f"received {profile_array.shape}"
        )

    with torch.no_grad():
        profile_tensor = torch.as_tensor(
            profile_array, dtype=torch.float32, device=device
        )

        mu, logvar = model.encode(profile_tensor)

        if use_mean_only:
            z_tensor = mu
        else:
            std = torch.exp(0.5 * torch.clamp(logvar, -12, 8))
            z_tensor = mu + torch.randn_like(std) * std

        z_latent = z_tensor.squeeze(0).cpu().numpy()

    return {
        "seed_amount": seed_amount,
        "z_latent": z_latent
    }

## Reconstruction examples and checks
Examples use held-out synthetic profiles. Scatter plots compare both the learned time coordinates and profile values. These are reconstruction diagnostics, not validation of physical executability.

In [ ]:
model.eval()
with torch.no_grad():
    reconstructed = model(synthetic_dataset_VAL)[0].cpu().numpy()
actual = synthetic_dataset_VAL.cpu().numpy()
assert reconstructed.shape == actual.shape == (N_sampl_VAL, 2, SEQ_LEN)
assert np.isfinite(reconstructed).all()
assert np.all(np.diff(reconstructed[:, 0], axis=-1) > 0)
assert np.allclose(reconstructed[:, 0, 0], 0)
assert np.allclose(reconstructed[:, 0, -1], 1)
assert np.all((reconstructed[:, 1] >= 0) & (reconstructed[:, 1] <= 1))
errors = np.abs(actual - reconstructed).mean(axis=(1, 2))
print(f"Mean absolute error (normalized coordinates): {errors.mean():.5f}")
print(f"Profiles with MAE > 0.05: {(errors > 0.05).mean():.1%}")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, index in zip(axes, np.random.choice(len(actual), size=5, replace=False)):
    ax.plot(actual[index, 0], actual[index, 1], 'o-', label='Original')
    ax.plot(reconstructed[index, 0], reconstructed[index, 1], 'o--', label='Reconstructed')
    ax.set(title=f'MAE {errors[index]:.4f}', xlabel='Normalized time', ylabel='Normalized profile', ylim=(-0.05, 1.05))
    ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for row, (ax, name) in enumerate(zip(axes, ['Time', 'Profile'])):
    a, r = actual[:, row].ravel(), reconstructed[:, row].ravel()
    ax.scatter(a, r, s=5, alpha=0.1)
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set(title=f'{name}: MAE {np.abs(a-r).mean():.4f}', xlabel='Actual', ylabel='Reconstructed')
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

example = actual[0]  # (2, SEQ_LEN); (1, 2, SEQ_LEN) is also accepted
encoded = encode_profile_for_bo(seed_amount=1.0, profile_trajectory=example)
decoded = decode_bo_suggestion(encoded['seed_amount'], encoded['z_latent'])
assert decoded['executable_profile_trajectory'].shape == (2, SEQ_LEN)
print('Latent shape:', encoded['z_latent'].shape)
print('Decoded trajectory shape:', decoded['executable_profile_trajectory'].shape)

## Use your own `(t, Profile)` trajectory
Use exactly `SEQ_LEN` ordered control points. Normalize time as `(t - t[0]) / (t[-1] - t[0])`. Normalize values using fixed bounds: `(Profile - PROFILE_MIN) / (PROFILE_MAX - PROFILE_MIN)`. Apply the same bounds to every training, validation and inference profile; choose bounds appropriate for the selected process variable.

Stack with `np.stack([t_normalized, profile_normalized])` and pass to `encode_profile_for_bo`. Restore physical values with `decoded["Profile"] * (PROFILE_MAX - PROFILE_MIN) + PROFILE_MIN`; restore time with `decoded["t"] * duration + start_time`. The normalized model does not learn absolute duration or physical bounds. Train separate models if using different profile families or scaling conventions.